In [ ]:
REPO_URL  = "https://github.com/nardouhn/ai-translator-system.git"
!git clone {REPO_URL}

In [ ]:
!pip install pyngrok flask-ngrok transformers accelerate bitsandbytes

In [ ]:
%cd /kaggle/working/ai-translator-system


In [ ]:
!git fetch
!git checkout feature/rag-module

In [ ]:
!pip install -q -r requirements.txt pyngrok flask accelerate

In [ ]:
# =========================
# 1. SETUP THƯ MỤC CỦA REPO ĐỂ RAG TÌM THẤY 'VectorDB_Gemini'
# =========================
import os
import sys

# Đổi thư mục làm việc vào thư mục source code vừa clone
if os.path.exists("ai-translator-system"):
    os.chdir("ai-translator-system")
    sys.path.append(os.getcwd())

# =========================
# 2. IMPORT GLOBAL & REPO LOCAL
# =========================
from flask import Flask, request, jsonify, Response
from transformers import AutoTokenizer, AutoModelForCausalLM
from pyngrok import ngrok
import torch
import threading
gpu_lock = threading.Lock()
import time

# IMPORT MODULE RAG TỪ REPOSITORY CỦA BẠN 
# (File prompt_config.py bạn viết bên trong nhánh feature/rag-module)
from prompt_config import get_context_prompt, format_qwen_prompt

# =========================
# 3. CONFIG
# =========================
NGROK_AUTH_TOKEN = "3CfCLgtblgbbqkVtBYsk9YLARVM_7knfsMBQXGYTMwQSeNSw5"

MODEL_ID = "ltyen05/qwen-domain-translator"
ALLOWED_DOMAINS = ["general", "it", "finance", "medical"]

# =========================
# 4. LOAD MODEL DỊCH (LLM)
# =========================
print("Loading Main Qwen Model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)
# Đưa vào chế độ eval để tăng tốc
model.eval()
print("Qwen Model loaded successfully!")

# =========================
# 5. CORE RAG FUNCTION
# =========================
cache = {}

def translate_text(text, domain):
    chunks = [text]
    results = []

    for chunk in chunks:
        rag_context = get_context_prompt(user_input=chunk, domain=domain)
        full_prompt = format_qwen_prompt(
            user_input=chunk,
            context=rag_context,
            domain=domain
        )

        inputs = tokenizer([full_prompt], return_tensors="pt").to(model.device)

        with torch.inference_mode():
            with gpu_lock:
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=False,
                    repetition_penalty=1.0,
                    pad_token_id=tokenizer.eos_token_id
                )

        generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
        result = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        results.append(result)

    return " ".join(results), False

# =========================
# 6. APP & API REST
# =========================
app = Flask(__name__)

# -- UI --
@app.route("/ui")
def ui():
    # Tương tự giao diện trên, tái sử dụng nhanh
    return """
    <html>
    <head><title>AI Translator (with RAG)</title></head>
    <body style="font-family: sans-serif; padding: 20px;">
        <h2>Kaggle Server - AI Translator (RAG Integrated)</h2>
        <textarea id="text" rows="5" cols="60" placeholder="Enter text in English..."></textarea><br><br>
        <select id="domain">
            <option value="general">General</option>
            <option value="it">IT</option>
            <option value="finance">Finance</option>
            <option value="medical">Medical</option>
        </select>
        <button onclick="send()">Translate</button>
        <h3>Result:</h3>
        <pre id="result" style="background: #f4f4f4; padding: 10px; border-radius: 5px; min-height: 20px;"></pre>
        <script>
        async function send() {
            document.getElementById("result").innerText = "Đang tìm kiếm database và dịch...";
            const text = document.getElementById("text").value;
            const domain = document.getElementById("domain").value;
            try {
                const res = await fetch("/translate", {
                    method: "POST", headers: {"Content-Type": "application/json"},
                    body: JSON.stringify({text, domain})
                });
                const data = await res.json();
                document.getElementById("result").innerText = data.output || data.error;
            } catch(e) {
                document.getElementById("result").innerText = "Error: " + e;
            }
        }
        </script>
    </body>
    </html>
    """

# -- MAIN API --
@app.route("/translate", methods=["POST"])
def translate():
    data = request.get_json()
    text = data.get("text", "").strip()
    domain = data.get("domain", "general")

    if not text:
        return jsonify({"error": "text required"}), 400
    if domain not in ALLOWED_DOMAINS:
        return jsonify({"error": "invalid domain"}), 400

    output, cached = translate_text(text, domain)
    return jsonify({"input": text, "domain": domain, "output": output, "cached": cached})

# =========================
# 7. CHẠY HỆ THỐNG
# =========================
def run_app():
    app.run(host="0.0.0.0", port=5000, use_reloader=False)

try:
    ngrok.kill()
except:
    pass

print("Starting ngrok...")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000)
url_str = getattr(public_url, "public_url", public_url)

print(f"\n🚀 PUBLIC URL (API & UI): {url_str}")
print(f"👉 UI Demo Link: {url_str}/ui\n")

threading.Thread(target=run_app, daemon=True).start()

while True:
    time.sleep(1000)


Loading Reranker model (ms-marco-MiniLM-L-6-v2) for multi-stage RAG...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Main Qwen Model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen Model loaded successfully!
Starting ngrok...

🚀 PUBLIC URL (API & UI): https://luncheon-scorebook-daylight.ngrok-free.dev
👉 UI Demo Link: https://luncheon-scorebook-daylight.ngrok-free.dev/ui

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit
127.0.0.1 - - [22/Apr/2026 16:46:51] "GET /ui HTTP/1.1" 200 -
/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 27.1MiB/s]
127.0.0.1 - - [22/Apr/2026 16:47:34] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [22/Apr/2026 16:48:52] "GET /ui HTTP/1.1" 200 -
127.0.0.1 - - [22/Apr/2026 16:50:57] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [22/Apr/2026 16:55:42] "GET /ui HTTP/1.1" 200 -
127.0.0.1 - - [22/Apr/2026 16:55:52] "GET /translate HTTP/1.1" 405 -
127.0.0.1 - - [22/Apr/2026 16:56:22] "POST /translate HTTP/1.1" 200 -
127.0.0.1 - - [22/Apr/2026 16:56:46] "GET /translate HTTP/1.1" 405 -
127.0.0.1 - - [22/Apr/2026 16:56:46] "GET /favicon.ico HTTP/1.1" 404 -
